In [1]:
import pandas as pd
import numpy as np

import os, sys, gc, time, warnings, pickle, psutil, random

from tqdm import tqdm
from math import ceil
from sklearn.preprocessing import LabelEncoder


In [2]:
grid_df = pd.read_parquet('train_val.parquet')
grid_df = grid_df.dropna(subset=['demand'])

In [3]:
grid_df

,id,item_id,store_id,state_id,dept_id,cat_id,d,demand,lag_t7,rolling_mean_lag7_w7,...,wday,month,year,event_name_1,event_name_2,snap_CA,snap_TX,snap_WI,sell_price,disaster_severity_flag
0,HOBBIES_1_001_CA_1,1438,1,1,4,2,1029,0.0,0.0,1.000000,...,7,11,2013,1,1,0,0,0,8.26,0
1,HOBBIES_1_002_CA_1,1439,1,1,4,2,1029,0.0,1.0,0.142857,...,7,11,2013,1,1,0,0,0,3.97,0
2,HOBBIES_1_003_CA_1,1440,1,1,4,2,1029,0.0,0.0,0.000000,...,7,11,2013,1,1,0,0,0,NaN,0
3,HOBBIES_1_004_CA_1,1441,1,1,4,2,1029,3.0,0.0,1.142857,...,7,11,2013,1,1,0,0,0,4.64,0
4,HOBBIES_1_005_CA_1,1442,1,1,4,2,1029,1.0,1.0,0.428571,...,7,11,2013,1,1,0,0,0,3.08,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27837365,FOODS_3_823_WI_3,1433,10,3,3,1,1941,1.0,3.0,0.714286,...,2,5,2016,1,1,0,0,0,2.98,0
27837366,FOODS_3_824_WI_3,1434,10,3,3,1,1941,0.0,0.0,0.000000,...,2,5,2016,1,1,0,0,0,2.48,0
27837367,FOODS_3_825_WI_3,1435,10,3,3,1,1941,2.0,1.0,1.142857,...,2,5,2016,1,1,0,0,0,3.98,0
27837368,FOODS_3_826_WI_3,1436,10,3,3,1,1941,0.0,1.0,1.000000,...,2,5,2016,1,1,0,0,0,1.28,0


In [4]:
print(grid_df.columns)

Index(['id', 'item_id', 'store_id', 'state_id', 'dept_id', 'cat_id', 'd',
       'demand', 'lag_t7', 'rolling_mean_lag7_w7', 'rolling_mean_lag7_w28',
       'lag_t28', 'rolling_mean_lag28_w7', 'rolling_mean_lag28_w28', 'wday',
       'month', 'year', 'event_name_1', 'event_name_2', 'snap_CA', 'snap_TX',
       'snap_WI', 'sell_price', 'disaster_severity_flag'],
      dtype='object')


In [5]:
m = grid_df.id=='HOBBIES_1_288_CA_1'
sales_ts = grid_df.loc[m,'demand'].values
sales_ts

array([ 7.,  1.,  4.,  5.,  6.,  3.,  0.,  0.,  7.,  2.,  6.,  2.,  0.,
        5.,  4.,  3.,  0.,  0.,  0.,  0.,  0.,  0., 14.,  4.,  1.,  1.,
        1.,  5.,  6.,  1.,  6., 14.,  1.,  0.,  0.,  4.,  2.,  4.,  7.,
        3.,  1.,  2.,  3.,  6.,  4.,  2.,  0.,  1.,  4.,  1.,  2.,  1.,
       11.,  2.,  1.,  3.,  5., 10.,  2.,  4.,  7.,  2.,  0.,  0.,  0.,
        4.,  0.,  8., 12.,  0.,  0.,  0.,  5.,  2., 14.,  0.,  0.,  1.,
        2.,  7.,  1.,  8.,  4.,  3.,  2.,  0.,  2., 13.,  4.,  2.,  0.,
       13.,  4.,  5.,  0.,  9.,  6.,  6.,  3.,  2.,  6.,  0.,  0.,  6.,
        0.,  4.,  0.,  3., 11.,  0.,  0.,  2., 10.,  7.,  4.,  2.,  5.,
        0.,  1.,  3.,  1.,  4.,  2.,  3., 24.,  0.,  3., 24.,  2.,  4.,
       10.,  2., 22., 11.,  8.,  0.,  8.,  7.,  2.,  6.,  9.,  7.,  6.,
        7.,  1.,  8.,  0.,  2.,  5., 12.,  0.,  0.,  0., 14.,  3.,  2.,
       13.,  3.,  5.,  0.,  4., 15.,  0.,  0.,  0.,  0.,  0.,  5.,  0.,
        6.,  3.,  4.,  0.,  0.,  0.,  4.,  0.,  0.,  0.,  0.,  0

In [6]:
def gap_finder(ts):
    
    # this function finds gaps and calculates their length:
    ts = (~(ts > 0)).astype(int)

    for i, val in enumerate(ts):
        if val == 0: 
            continue
        else: 
            ts[i] += ts[i-1]
            ts[i-1] = -1
    return ts

def gap_counter(ts):
    
    # value_counts for gaps lengths
    
    counts = np.unique(ts, return_counts=True)
    return dict(zip(counts[0], counts[1]))

m = grid_df.id=='HOBBIES_1_288_CA_1'

sales_gaps = gap_counter(gap_finder(grid_df.loc[m,'demand'].values))
sales_gaps

{-1: 410, 0: 375, 1: 67, 2: 35, 3: 14, 4: 4, 5: 3, 6: 2, 9: 1, 10: 1, 169: 1}

In [7]:
# Lets try to build a synthetic series with gaps.
# We assume products have a constant chance of being sold on any particular day.
# So for every day we will flip an unfair coin with probability of sale equal to 'dates_with sale'/'all_dates'
# >>>This might be an oversimplistic assumption, a moving window might be used instead.

def synth_sales(gaps_dict, min_gap=1000):
    
    sum_sale_days = gaps_dict[0] # key '0' gives number of days with sales
    sum_days = sum(gaps_dict.values()) # sum of all keys - number of all days
    
    # Sum of days in gaps longer than minimum gap length:
    sum_gap_length = sum([k for k in [*sales_gaps.keys()] if k > min_gap])
    
    # Exlude all the gaps longer than min_gap_length from probability calculation:
    p = sum_sale_days/(sum_days-sum_gap_length)
    
    return np.random.binomial(1, p, sum(gaps_dict.values()))

synth_sales_gaps = gap_counter(gap_finder(synth_sales(sales_gaps)))
synth_sales_gaps

{-1: 532,
 0: 166,
 1: 86,
 2: 58,
 3: 28,
 4: 18,
 5: 8,
 6: 5,
 7: 3,
 8: 2,
 9: 5,
 10: 1,
 12: 1}

In [8]:
# As you can see we dont have results > 20. 
# To make sure this is a consistent result lets run simulation 1,000 times:
# >>>> n equal 1,000 might be an overshot for some ids.

n=1000

sym_df = pd.DataFrame([gap_counter(gap_finder(synth_sales(sales_gaps))) for i in range(n)])
gap_length_prob = (sym_df.sum(axis=0)/n).sort_index()
gap_length_prob

-1     537.438
 0     154.835
 1      90.278
 2      53.972
 3      31.390
 4      18.672
 5      10.775
 6       6.320
 7       3.820
 8       2.268
 9       1.397
 10      0.783
 11      0.431
 12      0.271
 13      0.148
 14      0.086
 15      0.049
 16      0.027
 17      0.012
 18      0.010
 19      0.007
 20      0.005
 21      0.003
 22      0.002
 25      0.001
dtype: float64

In [9]:
# The shortest gap that has been seen in less than 5% of simulated series:
min_gap_length = min(gap_length_prob[gap_length_prob<0.05].index)
min_gap_length

15

In [10]:
# Now we exclude the gaps longer than min_gap_length from 'probability of sale' calculation, 
# because we assume them to be non-random.
# Run the simulation recursively until the min_gap_length does not decrease:

n=1000
new_min_gap_length = 0

while new_min_gap_length != min_gap_length:
    
    if new_min_gap_length!=0: min_gap_length=new_min_gap_length

    sym_df = pd.DataFrame([gap_counter(gap_finder(synth_sales(sales_gaps, min_gap_length))) for i in range(n)])
    gap_length_prob = (sym_df.sum(axis=0)/n).sort_index()

    # Lets find the shortes gap that has been seen in les than 5% of simulated series:
    new_min_gap_length = min(gap_length_prob[gap_length_prob<0.05].index)

min_gap_length

13

In [11]:
# Finally we need to make a feature 'out_of_stock' for the product:
m = grid_df.id=='HOBBIES_1_288_CA_1'
idx = grid_df.loc[m,'demand'].index

gf = pd.Series(gap_finder(grid_df.loc[m,'demand'].values), index = idx)
gf = gf.replace(-1,np.nan).fillna(method='backfill') 
gf = gf > min_gap_length
gf

C:\Users\ASUS\AppData\Local\Temp\ipykernel_15344\2685504888.py:6: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  gf = gf.replace(-1,np.nan).fillna(method='backfill')


280         False
30770       False
61260       False
91750       False
122240      False
            ...  
27685200    False
27715690    False
27746180    False
27776670    False
27807160    False
Length: 913, dtype: bool

In [ ]:
# Let calculate 'out-of-stock' for the first 30 ids:
grid_df['out_of_stock'] = 0

n=1000
prods = list(grid_df.id.unique())
gap_length_list = []

for prod_id in tqdm(prods):
    
    m = grid_df.id == prod_id
    idx = grid_df.loc[m,'demand'].index

    sales_gaps = gap_counter(gap_finder(grid_df.loc[m,'demand'].values))

    sym_df = pd.DataFrame([gap_counter(gap_finder(synth_sales(sales_gaps))) for i in range(n)])
    gap_length_prob = (sym_df.sum(axis=0)/n).sort_index()

    min_gap_length = min(gap_length_prob[gap_length_prob<0.05].index)
    new_min_gap_length = 0
    
    while new_min_gap_length < min_gap_length:

        if new_min_gap_length!=0: min_gap_length=new_min_gap_length

        sym_df = pd.DataFrame([gap_counter(gap_finder(synth_sales(sales_gaps, min_gap_length))) for i in range(n)])
        gap_length_prob = (sym_df.sum(axis=0)/n).sort_index()

        # Lets find the shortes gap that has been seen in les than 5% of simulated series:
        new_min_gap_length = min(gap_length_prob[gap_length_prob<0.05].index)

    gf = pd.Series(gap_finder(grid_df.loc[m,'demand'].values), index = idx)
    gf = gf.replace(-1,np.nan).fillna(method='backfill') 
    gf = gf > min_gap_length
    grid_df.loc[m,'out_of_stock'] = gf*1
    gap_length_list += [min_gap_length]

  0%|          | 0/30490 [00:00<?, ?it/s]C:\Users\ASUS\AppData\Local\Temp\ipykernel_15344\2179007503.py:32: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  gf = gf.replace(-1,np.nan).fillna(method='backfill')
  0%|          | 1/30490 [00:00<8:01:44,  1.05it/s]C:\Users\ASUS\AppData\Local\Temp\ipykernel_15344\2179007503.py:32: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  gf = gf.replace(-1,np.nan).fillna(method='backfill')
  0%|          | 2/30490 [00:01<7:10:17,  1.18it/s]C:\Users\ASUS\AppData\Local\Temp\ipykernel_15344\2179007503.py:32: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  gf = gf.replace(-1,np.nan).fillna(method='backfill')
  0%|          | 3/30490 [00:02<6:49:47,  1.24it/s]C:\Users\ASUS\AppData\Local\Temp\ipykerne